# L4 创作实验室：想法 → 文案 → 图片 → Wan 视频
**目标配置：NVIDIA L4 24GB、64GB 系统内存、200GB 硬盘。** 无需接入作者或任何人的 Colab。

**验证边界：作者未在 L4/Colab 实际执行，未生成示例视频，未测量峰值显存和耗时。**
已核对官方仓库、模型卡、CLI 参数及权重目录，并附格式/语法/API 模拟检查报告。
笔记本会在你的运行时执行真正的 CUDA、FlashAttention 小算子、导入和输出文件自检；这些不是端到端成功保证。

- 文案：`Qwen/Qwen2.5-1.5B-Instruct`（BF16，无付费 API）。
- 文生图：`stabilityai/sd-turbo`（FP16、512×512、4 步、CFG=0）。保留原图，并展示裁切放大的实际视频首帧。
- 图生视频：官方 `Wan-AI/Wan2.2-TI2V-5B`，官方 `generate.py`，不使用 ComfyUI 或自定义节点。
- 各阶段独立子进程，退出后释放 CUDA 上下文；文件锁避免网页与笔记本同时占用 GPU。
- 默认只执行笔记本；网页和公网开关均关闭。可选 FastAPI 页面及 API 统一 HTTP Basic 鉴权，任务通过轮询展示阶段结果。

**先选择「运行时 → 更改运行时类型 → L4 GPU」，并启用高内存。** 从上到下运行 1–8。
默认 33 帧/20 步用于短片试跑（约 1.38 秒）；跑通后可改 121 帧/50 步（约 5.04 秒）。
1280×704 和 24 FPS 来自官方配置；减少帧数和步数是本笔记本的试跑设置，画质/运动不代表官方标准效果。
官方说明该卸载配置面向至少 24GB 显存；L4 与 4090 的速度不同，不提供固定完成时间承诺。


## 1 · 配置与硬件检查
使用独立 Python 3.11 环境，不替换 Colab 的 PyTorch。安装、权重缓存和产物都在 `/content/wan_l4_lab`。
总硬盘容量不等于可用容量；安装前检查至少 100 GiB 空闲，系统内存至少 40 GiB 可用。
不自动挂载 Drive，不自动开启隧道。Colab 断开/回收会丢失 `/content`，请及时下载结果。


In [ ]:
import os, sys, json, subprocess, shutil, time, signal, base64, getpass, urllib.request
from pathlib import Path
from IPython.display import display, Image, Video, JSON
BASE = Path('/content/wan_l4_lab')
BASE.mkdir(parents=True, exist_ok=True)
os.environ['WAN_LAB_HOME'] = str(BASE)
ENABLE_WEB = False       # 改 True 才启动带密码网页/API
ENABLE_PUBLIC = False    # 改 True 且网页已启动，才创建 Cloudflare 临时公网地址
IDEA = '一只金毛犬在晨光中的草地上慢慢奔跑，野花轻轻摇动，温暖电影感。'
SEED, FRAMES, STEPS = 42, 33, 20
# 正式效果可用：FRAMES, STEPS = 121, 50；请先完成默认短片试跑。
def run_logged(cmd, name, env=None, cwd=None):
    """实时展示输出并留日志；失败立即停止，避免继续运行后续阶段。"""
    with (BASE / (name + '.log')).open('w') as log:
        p = subprocess.Popen([str(x) for x in cmd], cwd=cwd, env=env,
            stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True,
            bufsize=1, start_new_session=True)
        try:
            for line in p.stdout:
                log.write(line); log.flush(); print(line, end='')
            rc = p.wait()
        except BaseException:
            os.killpg(p.pid, signal.SIGTERM)
            try: p.wait(timeout=10)
            except subprocess.TimeoutExpired:
                os.killpg(p.pid, signal.SIGKILL); p.wait()
            raise
    if rc: raise RuntimeError(f'{name} 失败，查看 {BASE / (name + ".log")}')


In [ ]:
FILES = {'core.py': '"""CPU-only orchestration; model stages always live in separate processes."""\nimport contextlib\nimport fcntl\nimport json\nimport os\nfrom pathlib import Path\nimport signal\nimport subprocess\nimport time\nimport uuid\n\nBASE = Path(os.environ.get(\'WAN_LAB_HOME\', \'/content/wan_l4_lab\')).resolve()\nRUNS = BASE / \'runs\'\nPY = BASE / \'venv/bin/python\'\nREPO = BASE / \'Wan2.2\'\n\ndef load_config():\n    return json.loads((BASE / \'config.json\').read_text())\n\ndef new_run(idea, seed=42, frames=33, steps=20):\n    if not isinstance(idea, str) or not 1 <= len(idea.strip()) <= 2000:\n        raise ValueError(\'想法须为 1–2000 字符\')\n    if type(seed) is not int or not 0 <= seed <= 4294967295:\n        raise ValueError(\'seed 超出范围\')\n    if frames not in (33, 65, 121) or type(frames) is not int:\n        raise ValueError(\'frames 只能为 33、65、121（4n+1）\')\n    if type(steps) is not int or not 10 <= steps <= 50:\n        raise ValueError(\'steps 须为 10–50 的整数\')\n    run = RUNS / uuid.uuid4().hex\n    run.mkdir(parents=True)\n    (run / \'request.json\').write_text(json.dumps(dict(idea=idea.strip(), seed=seed,\n        frames=frames, steps=steps), ensure_ascii=False, indent=2))\n    return run\n\ndef read_plan(run):\n    plan = json.loads((Path(run) / \'plan.json\').read_text())\n    for key, limit in [(\'copy_zh\', 2000), (\'image_prompt\', 1800), (\'video_prompt\', 1800)]:\n        if not isinstance(plan.get(key), str) or not 1 <= len(plan[key].strip()) <= limit:\n            raise ValueError(\'文案 JSON 缺少有效字段：\' + key)\n    return plan\n\n@contextlib.contextmanager\ndef gpu_lock():\n    BASE.mkdir(parents=True, exist_ok=True)\n    with (BASE / \'gpu.lock\').open(\'a\') as handle:\n        try:\n            fcntl.flock(handle, fcntl.LOCK_EX | fcntl.LOCK_NB)\n        except BlockingIOError:\n            raise RuntimeError(\'GPU 正忙；请等待当前阶段完成\')\n        try:\n            yield\n        finally:\n            fcntl.flock(handle, fcntl.LOCK_UN)\n\ndef video_command(run, config=None):\n    run = Path(run).resolve()\n    c = config or load_config()\n    req = json.loads((run / \'request.json\').read_text())\n    plan = read_plan(run)\n    if not (run / \'image.png\').is_file():\n        raise FileNotFoundError(\'先生成首帧 image.png\')\n    return [str(PY), str(REPO / \'generate.py\'), \'--task\', \'ti2v-5B\',\n        \'--size\', \'1280*704\', \'--ckpt_dir\', c[\'models\'][\'wan\'],\n        \'--offload_model\', \'True\', \'--convert_model_dtype\', \'--t5_cpu\',\n        \'--image\', str(run / \'image.png\'), \'--prompt\', plan[\'video_prompt\'],\n        \'--frame_num\', str(req[\'frames\']), \'--sample_steps\', str(req[\'steps\']),\n        \'--sample_solver\', \'unipc\', \'--sample_shift\', \'5.0\',\n        \'--sample_guide_scale\', \'5.0\', \'--base_seed\', str(req[\'seed\']),\n        \'--save_file\', str(run / \'video.mp4\')]\n\ndef terminate_tree(proc):\n    if proc.poll() is None:\n        os.killpg(proc.pid, signal.SIGTERM)\n        try:\n            proc.wait(timeout=10)\n        except subprocess.TimeoutExpired:\n            os.killpg(proc.pid, signal.SIGKILL)\n            proc.wait()\n\ndef run_stage(stage, run, timeout=7200):\n    if stage not in (\'text\', \'image\', \'video\'):\n        raise ValueError(\'未知阶段\')\n    run = Path(run).resolve()\n    if run.parent != RUNS or not (run / \'request.json\').is_file():\n        raise ValueError(\'无效运行目录\')\n    with gpu_lock():\n        cmd = video_command(run) if stage == \'video\' else [str(PY),\n            str(BASE / \'worker.py\'), stage, str(run)]\n        env = dict(os.environ, PYTHONUNBUFFERED=\'1\', TOKENIZERS_PARALLELISM=\'false\',\n            HF_HUB_OFFLINE=\'1\', TRANSFORMERS_OFFLINE=\'1\',\n            PYTORCH_CUDA_ALLOC_CONF=\'expandable_segments:True\')\n        # Do not pass web credentials or optional Hub token into model processes.\n        for key in (\'LAB_PASSWORD\', \'LAB_USER\', \'HF_TOKEN\'):\n            env.pop(key, None)\n        log = run / (stage + \'.log\')\n        start = time.monotonic()\n        with log.open(\'w\') as out:\n            proc = subprocess.Popen(cmd, cwd=REPO, env=env, stdout=out,\n                stderr=subprocess.STDOUT, start_new_session=True)\n            try:\n                code = proc.wait(timeout=timeout)\n            finally:\n                terminate_tree(proc)\n        if code:\n            raise RuntimeError(f\'{stage} 失败（exit={code}），查看 {log.name}\')\n        expected = {\'text\': \'plan.json\', \'image\': \'image.png\', \'video\': \'video.mp4\'}[stage]\n        if not (run / expected).is_file() or (run / expected).stat().st_size == 0:\n            raise RuntimeError(\'阶段结束但未生成文件：\' + expected)\n        if stage == \'text\':\n            read_plan(run)\n        if stage == \'video\':\n            probe = subprocess.run([\'ffprobe\', \'-v\', \'error\', \'-select_streams\', \'v:0\',\n                \'-show_entries\', \'stream=width,height,nb_frames,r_frame_rate,duration\',\n                \'-of\', \'json\', str(run / expected)], capture_output=True, text=True, check=True)\n            info = json.loads(probe.stdout)[\'streams\'][0]\n            req = json.loads((run / \'request.json\').read_text())\n            if (int(info[\'width\']), int(info[\'height\'])) != (1280, 704):\n                raise RuntimeError(\'输出尺寸不符：\' + str(info))\n            if int(info.get(\'nb_frames\', -1)) != req[\'frames\'] or info[\'r_frame_rate\'] != \'24/1\':\n                raise RuntimeError(\'输出帧数或帧率不符：\' + str(info))\n            (run / \'video_info.json\').write_text(json.dumps(info, indent=2))\n        metric = {\'stage\': stage, \'seconds\': round(time.monotonic()-start, 2),\n                  \'note\': \'子进程已退出，CUDA 上下文随进程释放；不是峰值显存测量\'}\n        try:\n            metric[\'gpu_after\'] = subprocess.check_output([\'nvidia-smi\',\n                \'--query-gpu=memory.used,memory.free\', \'--format=csv,noheader\'], text=True).strip()\n        except (OSError, subprocess.SubprocessError):\n            pass\n        (run / (stage + \'_metrics.json\')).write_text(json.dumps(metric, ensure_ascii=False, indent=2))\n        return run / expected\n', 'worker.py': "import json\nfrom pathlib import Path\nimport sys\nfrom core import BASE, load_config, read_plan\n\ndef main(stage, folder):\n    import torch\n    run = Path(folder)\n    config = load_config()\n    req = json.loads((run / 'request.json').read_text())\n    torch.manual_seed(req['seed'])\n    if stage == 'text':\n        from transformers import AutoTokenizer, AutoModelForCausalLM\n        model_dir = config['models']['text']\n        tokenizer = AutoTokenizer.from_pretrained(model_dir, local_files_only=True,\n                                                 trust_remote_code=False)\n        model = AutoModelForCausalLM.from_pretrained(model_dir,\n            torch_dtype=torch.bfloat16, local_files_only=True,\n            trust_remote_code=False, use_safetensors=True).to('cuda').eval()\n        messages = [dict(role='system', content=(\n            'You are a video creative writer. Return ONLY one valid JSON object, '\n            'no markdown. Keys: copy_zh (short Chinese promotional copy), '\n            'image_prompt (English description of ONE initial frame), '\n            'video_prompt (English description of the SAME scene with gentle motion '\n            'and camera movement, no cuts). Keep subject, setting and lighting consistent. '\n            'No subtitles or rendered text. All values must be nonempty strings.')),\n            dict(role='user', content=req['idea'])]\n        text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)\n        inputs = tokenizer(text, return_tensors='pt').to('cuda')\n        with torch.inference_mode():\n            output = model.generate(**inputs, max_new_tokens=700, do_sample=False,\n                                    pad_token_id=tokenizer.eos_token_id)\n        raw = tokenizer.decode(output[0, inputs.input_ids.shape[1]:], skip_special_tokens=True)\n        (run / 'text_raw.txt').write_text(raw)\n        # No invented fallback prompts: malformed output stops with raw text saved for repair.\n        begin, end = raw.find('{'), raw.rfind('}')\n        if begin < 0 or end <= begin:\n            raise ValueError('模型没有返回 JSON，请检查 text_raw.txt 并修正 plan.json')\n        plan = json.loads(raw[begin:end+1])\n        (run / 'plan.json').write_text(json.dumps(plan, ensure_ascii=False, indent=2))\n        read_plan(run)\n        print(json.dumps(plan, ensure_ascii=False, indent=2))\n    elif stage == 'image':\n        from diffusers import AutoPipelineForText2Image\n        from PIL import Image, ImageOps\n        plan = read_plan(run)\n        pipe = AutoPipelineForText2Image.from_pretrained(config['models']['image'],\n            torch_dtype=torch.float16, variant='fp16', use_safetensors=True,\n            local_files_only=True).to('cuda')\n        pipe.enable_vae_slicing()\n        pipe.enable_vae_tiling()\n        with torch.inference_mode():\n            result = pipe(prompt=plan['image_prompt'], num_inference_steps=4,\n                guidance_scale=0.0, width=512, height=512,\n                generator=torch.Generator(device='cuda').manual_seed(req['seed']))\n        if result.nsfw_content_detected is not None and any(result.nsfw_content_detected):\n            raise ValueError('模型安全检查拦截了输出，请修改想法')\n        image = result.images[0].convert('RGB')\n        image.save(run / 'image_original.png')\n        # Display this exact crop before video generation; upscaling adds no detail.\n        ImageOps.fit(image, (1280, 704), method=Image.Resampling.LANCZOS).save(run / 'image.png')\n        print('已保存原始 512×512 图和裁切放大的 1280×704 视频首帧')\n    else:\n        raise ValueError(stage)\n\nif __name__ == '__main__':\n    main(*sys.argv[1:])\n", 'server.py': '"""Single-user demo: HTTP Basic auth guards every route, including media."""\nimport base64\nimport binascii\nimport json\nimport os\nfrom pathlib import Path\nimport re\nimport secrets\nimport threading\nimport time\nfrom urllib.parse import urlsplit\nfrom fastapi import FastAPI, HTTPException\nfrom fastapi.responses import HTMLResponse, FileResponse, JSONResponse\nfrom pydantic import BaseModel, ConfigDict, Field\nfrom core import BASE, RUNS, new_run, read_plan, run_stage\n\nUSER = os.environ.get(\'LAB_USER\', \'owner\')\nPASSWORD = os.environ.get(\'LAB_PASSWORD\', \'\')\nif len(PASSWORD) < 16:\n    raise RuntimeError(\'LAB_PASSWORD 至少 16 字符；使用启动单元格中的隐藏输入\')\napp = FastAPI(docs_url=None, redoc_url=None, openapi_url=None)\njobs = {}\nstate_lock = threading.Lock()\nadmission = threading.Lock()\nFILES = {\'image.png\', \'image_original.png\', \'video.mp4\', \'plan.json\'}\n\nclass Guard:\n    def __init__(self, app):\n        self.app = app\n\n    async def __call__(self, scope, receive, send):\n        if scope[\'type\'] != \'http\':\n            # This app uses polling; no websocket endpoints are needed.\n            if scope[\'type\'] == \'websocket\':\n                await send({\'type\': \'websocket.close\', \'code\': 1008})\n                return\n            return await self.app(scope, receive, send)\n        headers = dict(scope.get(\'headers\', []))\n        authenticated = False\n        try:\n            kind, value = headers.get(b\'authorization\', b\'\').decode().split(\' \', 1)\n            if kind.lower() == \'basic\':\n                user, password = base64.b64decode(value, validate=True).decode(\'utf-8\').split(\':\', 1)\n                u = secrets.compare_digest(user.encode(), USER.encode())\n                p = secrets.compare_digest(password.encode(), PASSWORD.encode())\n                authenticated = u & p\n        except (ValueError, UnicodeError, binascii.Error):\n            pass\n        if not authenticated:\n            response = JSONResponse({\'detail\': \'Authentication required\'}, status_code=401,\n                headers={\'WWW-Authenticate\': \'Basic realm="Wan L4", charset="UTF-8"\',\n                         \'Cache-Control\': \'no-store\'})\n            return await response(scope, receive, send)\n        if scope[\'method\'] in (\'POST\', \'PUT\', \'PATCH\', \'DELETE\'):\n            origin = headers.get(b\'origin\', b\'\').decode()\n            if origin and urlsplit(origin).netloc != headers.get(b\'host\', b\'\').decode():\n                return await JSONResponse({\'detail\': \'Origin rejected\'}, 403)(scope, receive, send)\n            if headers.get(b\'content-type\', b\'\').split(b\';\')[0] != b\'application/json\':\n                return await JSONResponse({\'detail\': \'JSON required\'}, 415)(scope, receive, send)\n            chunks, total = [], 0\n            while True:\n                msg = await receive()\n                if msg[\'type\'] == \'http.disconnect\':\n                    return\n                total += len(msg.get(\'body\', b\'\'))\n                if total > 16384:\n                    return await JSONResponse({\'detail\': \'Body too large\'}, 413)(scope, receive, send)\n                chunks.append(msg.get(\'body\', b\'\'))\n                if not msg.get(\'more_body\', False):\n                    break\n            delivered = False\n            original_receive = receive\n            async def replay():\n                nonlocal delivered\n                if not delivered:\n                    delivered = True\n                    return {\'type\': \'http.request\', \'body\': b\'\'.join(chunks), \'more_body\': False}\n                return await original_receive()\n            receive = replay\n        async def safe_send(message):\n            if message[\'type\'] == \'http.response.start\':\n                message[\'headers\'] = list(message.get(\'headers\', [])) + [\n                    (b\'cache-control\', b\'no-store\'), (b\'x-content-type-options\', b\'nosniff\'),\n                    (b\'x-frame-options\', b\'DENY\'), (b\'referrer-policy\', b\'no-referrer\')]\n            await send(message)\n        await self.app(scope, receive, safe_send)\n\napp.add_middleware(Guard)\n\nclass JobRequest(BaseModel):\n    model_config = ConfigDict(extra=\'forbid\', strict=True)\n    idea: str = Field(min_length=1, max_length=2000)\n    seed: int = Field(default=42, ge=0, le=4294967295)\n    frames: int = 33\n    steps: int = Field(default=20, ge=10, le=50)\n\ndef snapshot(job_id):\n    if not re.fullmatch(r\'[0-9a-f]{32}\', job_id):\n        raise HTTPException(404)\n    with state_lock:\n        if job_id not in jobs:\n            raise HTTPException(404)\n        return json.loads(json.dumps(jobs[job_id]))\n\ndef update(job_id, **changes):\n    with state_lock:\n        jobs[job_id].update(changes)\n\ndef execute(run):\n    job_id = run.name\n    try:\n        for stage in (\'text\', \'image\', \'video\'):\n            update(job_id, stage=stage)\n            run_stage(stage, run)\n            if stage == \'text\':\n                update(job_id, plan=read_plan(run), text_ready=True)\n            elif stage == \'image\':\n                update(job_id, image_ready=True)\n            else:\n                update(job_id, video_ready=True)\n        update(job_id, status=\'done\', stage=\'complete\')\n    except Exception as error:\n        # Full details stay in local logs, no filesystem paths/stack traces in public API.\n        (run / \'error.txt\').write_text(repr(error))\n        update(job_id, status=\'error\', error=\'阶段失败；在 Colab 查看本次运行目录中的日志\')\n    finally:\n        admission.release()\n\n@app.get(\'/\', response_class=HTMLResponse)\ndef home():\n    return (BASE / \'index.html\').read_text()\n\n@app.get(\'/health\')\ndef health():\n    return {\'ok\': True, \'busy\': admission.locked(), \'real_gpu_verified\': False}\n\n@app.post(\'/jobs\', status_code=202)\ndef create_job(body: JobRequest):\n    if body.frames not in (33, 65, 121) or not body.idea.strip():\n        raise HTTPException(422, \'frames 只能为 33/65/121，想法不能为空\')\n    if not admission.acquire(blocking=False):\n        raise HTTPException(409, \'已有任务运行中\')\n    try:\n        with state_lock:\n            if len(jobs) >= 100:\n                raise HTTPException(429, \'本次服务已达 100 次运行，请保存结果并重启服务\')\n        run = new_run(body.idea, body.seed, body.frames, body.steps)\n        with state_lock:\n            jobs[run.name] = dict(id=run.name, status=\'running\', stage=\'text\',\n                text_ready=False, image_ready=False, video_ready=False)\n        threading.Thread(target=execute, args=(run,), daemon=True).start()\n        return {\'id\': run.name, \'status_url\': \'/jobs/\' + run.name}\n    except BaseException:\n        admission.release()\n        raise\n\n@app.get(\'/jobs/{job_id}\')\ndef get_job(job_id: str):\n    return snapshot(job_id)\n\n@app.get(\'/files/{job_id}/{name}\')\ndef get_file(job_id: str, name: str):\n    state = snapshot(job_id)\n    if name not in FILES:\n        raise HTTPException(404)\n    ready = state.get(\'video_ready\') if name == \'video.mp4\' else (\n        state.get(\'text_ready\') if name == \'plan.json\' else state.get(\'image_ready\'))\n    if not ready:\n        raise HTTPException(404)\n    path = RUNS / job_id / name\n    if not path.is_file() or path.is_symlink():\n        raise HTTPException(404)\n    media = \'video/mp4\' if name.endswith(\'.mp4\') else (\n        \'image/png\' if name.endswith(\'.png\') else \'application/json\')\n    return FileResponse(path, media_type=media)\n\nif __name__ == \'__main__\':\n    import uvicorn\n    uvicorn.run(app, host=\'127.0.0.1\', port=8000, access_log=False,\n                proxy_headers=False, workers=1)\n', 'index.html': '<!doctype html><html lang="zh-CN"><meta charset="utf-8">\n<meta name="viewport" content="width=device-width, initial-scale=1">\n<title>想法到短片 · Wan L4</title>\n<style>\nbody{font:16px/1.6 system-ui;background:#f5f3ef;color:#24312e;margin:32px auto;padding:0 20px;max-width:900px}\nh1{font-size:30px}section{background:white;border-radius:16px;padding:24px;margin:20px 0}\ntextarea{box-sizing:border-box;width:100%;padding:12px;font:inherit}button{padding:12px 24px;background:#245f50;color:white;border:0;border-radius:8px;cursor:pointer}\nbutton:disabled{opacity:.5}input,select{padding:8px;margin:8px}pre{white-space:pre-wrap;overflow-wrap:anywhere}img,video{max-width:100%;border-radius:10px}small{color:#53635e}\n</style>\n<h1>想法 → 文案 → 首帧 → 视频</h1>\n<p>Qwen 1.5B · SD-Turbo · 官方 Wan2.2 TI2V 5B。每阶段独立释放显存。</p>\n<section><label for="idea">描述一个单镜头想法</label>\n<textarea id="idea" rows="3" maxlength="2000">一只金毛犬在晨光中的草地上慢慢奔跑，野花轻轻摇动，温暖电影感。</textarea>\n<label>Seed<input id="seed" type="number" min="0" max="4294967295" value="42"></label>\n<label>帧数<select id="frames"><option>33</option><option>65</option><option>121</option></select></label>\n<label>步数<input id="steps" type="number" min="10" max="50" value="20"></label>\n<p><button id="go">生成短片</button> <span id="status">尚未开始</span></p>\n<small>33 帧约 1.38 秒；121 帧约 5.04 秒（24 FPS）。短片生成可能较慢，请保留 Colab 运行连接。</small></section>\n<section><h2>1 · 文案与提示词</h2><pre id="plan">等待生成</pre></section>\n<section><h2>2 · 视频首帧</h2><p>原图为 512×512；此处展示裁切放大的 1280×704 实际输入。</p><img id="image" hidden alt="生成的视频首帧"></section>\n<section><h2>3 · 视频结果</h2><video id="video" hidden controls playsinline preload="metadata"></video><p><a id="download" hidden>下载 MP4</a></p></section>\n<script>\nconst $=id=>document.getElementById(id); let timer=null,current=null;\nasync function get(url,options){const r=await fetch(url,options);if(!r.ok)throw Error(\'HTTP \'+r.status+\': \'+await r.text());return r.json()}\nasync function poll(){try{const j=await get(\'/jobs/\'+current);$(\'status\').textContent=j.status+\' · \'+j.stage;\nif(j.text_ready)$(\'plan\').textContent=JSON.stringify(j.plan,null,2);\nif(j.image_ready&&$(\'image\').hidden){$(\'image\').src=\'/files/\'+current+\'/image.png\';$(\'image\').hidden=false}\nif(j.video_ready&&$(\'video\').hidden){const u=\'/files/\'+current+\'/video.mp4\';$(\'video\').src=u;$(\'video\').hidden=false;$(\'download\').href=u;$(\'download\').download=\'wan-video.mp4\';$(\'download\').hidden=false}\nif(j.status===\'error\')$(\'status\').textContent=j.error;\nif(j.status===\'running\'){timer=setTimeout(poll,4000)}else{$(\'go\').disabled=false;sessionStorage.removeItem(\'wan_job\')}\n}catch(e){$(\'status\').textContent=\'连接中断，可稍后刷新重试：\'+e.message;$(\'go\').disabled=false}}\n$(\'go\').onclick=async()=>{clearTimeout(timer);$(\'go\').disabled=true;$(\'image\').hidden=true;$(\'video\').hidden=true;$(\'video\').removeAttribute(\'src\');$(\'download\').hidden=true;$(\'plan\').textContent=\'等待生成\';\ntry{const j=await get(\'/jobs\',{method:\'POST\',headers:{\'Content-Type\':\'application/json\'},body:JSON.stringify({idea:$(\'idea\').value,seed:Number($(\'seed\').value),frames:Number($(\'frames\').value),steps:Number($(\'steps\').value)})});current=j.id;sessionStorage.setItem(\'wan_job\',current);poll()}\ncatch(e){$(\'status\').textContent=e.message;$(\'go\').disabled=false}};\ncurrent=sessionStorage.getItem(\'wan_job\');if(current){$(\'go\').disabled=true;poll()}\n</script></html>\n', 'preflight.py': "import json\nimport os\nfrom pathlib import Path\nimport shutil\nimport subprocess\nimport sys\n\ndef host_check(base, before_install=False):\n    if sys.platform != 'linux':\n        raise RuntimeError('请在 Linux Google Colab GPU 运行时执行')\n    rows = subprocess.check_output(['nvidia-smi',\n        '--query-gpu=name,memory.total,memory.free,driver_version',\n        '--format=csv,noheader,nounits'], text=True).strip().splitlines()\n    name, total, free, driver = [x.strip() for x in rows[0].split(',')]\n    if 'L4' not in name:\n        raise RuntimeError('本笔记本针对 NVIDIA L4；当前为 ' + name)\n    if float(total) < 22000 or float(free) < 21000:\n        raise RuntimeError('显存不足或被占用，请用空闲 L4 24GB 运行时')\n    if tuple(map(int, driver.split('.')[:2])) < (550, 54):\n        raise RuntimeError('此固定 CUDA 12.4 环境要求驱动至少 550.54')\n    mem = {s.split(':')[0]: int(s.split()[1])*1024 for s in\n           Path('/proc/meminfo').read_text().splitlines()}\n    available = mem['MemAvailable']/2**30\n    disk = shutil.disk_usage(base).free/2**30\n    if available < 40:\n        raise RuntimeError(f'可用系统内存只有 {available:.1f} GiB；需高内存运行时，建议总内存 64GB')\n    if before_install and disk < 100:\n        raise RuntimeError(f'可用硬盘仅 {disk:.1f} GiB；安装前建议至少 100 GiB')\n    print(json.dumps(dict(gpu=name, vram_total_mib=total, vram_free_mib=free,\n        driver=driver, available_ram_gib=round(available,1), free_disk_gib=round(disk,1)), indent=2))\n\ndef gpu_check():\n    import torch\n    import flash_attn\n    import wan\n    from wan.configs import WAN_CONFIGS, SUPPORTED_SIZES\n    assert torch.__version__.startswith('2.6.0')\n    assert torch.cuda.is_available() and torch.cuda.is_bf16_supported()\n    assert torch._C._GLIBCXX_USE_CXX11_ABI is False, 'FlashAttention wheel ABI 不匹配'\n    from flash_attn import flash_attn_func\n    q = torch.randn(1, 32, 4, 64, device='cuda', dtype=torch.bfloat16)\n    y = flash_attn_func(q, q, q)\n    assert y.shape == q.shape and torch.isfinite(y).all()\n    assert '1280*704' in SUPPORTED_SIZES['ti2v-5B']\n    cfg = WAN_CONFIGS['ti2v-5B']\n    assert (cfg.sample_fps, cfg.sample_steps, cfg.frame_num) == (24, 50, 121)\n    print('CUDA、BF16、FlashAttention2 实际小算子、Wan 导入及任务配置检查通过')\n\nif __name__ == '__main__':\n    base = Path(os.environ.get('WAN_LAB_HOME', '/content/wan_l4_lab'))\n    host_check(base)\n    gpu_check()\n", 'download_models.py': "import json\nimport os\nfrom fnmatch import fnmatch\nfrom pathlib import Path\nimport shutil\nfrom core import BASE\n\n# Set before importing Hugging Face/Xet so cache stays with this Colab project.\nos.environ.setdefault('HF_XET_CACHE', str(BASE / 'hf_cache' / 'xet'))\nfrom huggingface_hub import HfApi, snapshot_download, try_to_load_from_cache\nfrom huggingface_hub.utils._runtime import is_xet_available\n\nIGNORE = {'image': ['sd_turbo.safetensors', 'text_encoder/model.safetensors',\n    'unet/diffusion_pytorch_model.safetensors', 'vae/diffusion_pytorch_model.safetensors']}\n\ndef wanted(name, filename):\n    return not any(fnmatch(filename, pattern) for pattern in IGNORE.get(name, []))\n\ndef download():\n    if not is_xet_available():\n        raise RuntimeError('下载环境缺少 hf-xet；在独立环境安装 hf-xet==1.8.0 后重跑第 4 节')\n    from hf_xet import PyPointerFile, download_files\n    print('Xet 客户端可用；使用 4 个文件下载线程。实际速度取决于网络与磁盘。', flush=True)\n    cfg = json.loads((BASE / 'config.json').read_text())\n    api = HfApi()\n    models, manifests = {}, {}\n    remaining = 0\n    for name, item in cfg['model_specs'].items():\n        info = api.model_info(item['repo'], revision=item['revision'], files_metadata=True)\n        if info.sha != item['revision']:\n            raise RuntimeError('模型版本不一致')\n        manifests[name] = {f.rfilename: f.size for f in info.siblings\n                          if f.size is not None and wanted(name, f.rfilename)}\n        for relative, size in manifests[name].items():\n            cached = try_to_load_from_cache(item['repo'], relative,\n                revision=item['revision'], cache_dir=BASE / 'hf_cache')\n            if isinstance(cached, str) and Path(cached).is_file() and Path(cached).stat().st_size == size:\n                continue\n            remaining += size\n    # Completed cached files are reused. Incomplete files are conservatively counted in full.\n    if shutil.disk_usage(BASE).free < remaining + 15*2**30:\n        raise RuntimeError(f'下载前空间检查失败：尚需下载约 {remaining/2**30:.1f} GiB，另留 15 GiB')\n    for name, item in cfg['model_specs'].items():\n        folder = Path(snapshot_download(item['repo'], revision=item['revision'],\n            cache_dir=BASE / 'hf_cache', max_workers=4,\n            ignore_patterns=IGNORE.get(name)))\n        for relative, size in manifests[name].items():\n            p = folder / relative\n            if not p.is_file() or p.stat().st_size != size:\n                raise RuntimeError('下载文件缺失或长度不符：' + relative)\n        models[name] = str(folder)\n        print(name, '固定版本下载及文件长度检查通过', flush=True)\n    cfg['models'] = models\n    (BASE / 'config.json').write_text(json.dumps(cfg, ensure_ascii=False, indent=2))\n    (BASE / 'model_manifest.json').write_text(json.dumps(manifests, indent=2))\n    # Check official format before allocating models.\n    wan = Path(models['wan'])\n    for rel in ['config.json', 'Wan2.2_VAE.pth', 'models_t5_umt5-xxl-enc-bf16.pth', 'google/umt5-xxl']:\n        if not (wan / rel).exists():\n            raise RuntimeError('官方 Wan 目录缺少：' + rel)\n    print('模型已就绪；后续生成阶段强制离线加载，不调用付费服务。')\n\nif __name__ == '__main__':\n    download()\n", 'requirements.txt': '# Python 3.11; torch/torchvision installed separately from official cu124 index.\ntorch==2.6.0\ntorchvision==0.21.0\ndiffusers==0.33.1\ntransformers==4.51.3\ntokenizers==0.21.1\naccelerate==1.6.0\nhuggingface-hub==0.30.2\nhf-xet==1.8.0\nsafetensors==0.5.3\nnumpy==1.26.4\nopencv-python==4.11.0.86\nPillow==11.2.1\nimageio==2.37.0\nimageio-ffmpeg==0.6.0\neinops==0.8.1\neasydict==1.13\nftfy==6.3.1\ndashscope==1.23.2\ntqdm==4.67.1\nfastapi==0.115.12\nstarlette==0.46.2\npydantic==2.11.4\nuvicorn==0.34.2\nhttpx==0.28.1\npackaging==25.0\n'}
for name, content in FILES.items():
    (BASE / name).write_text(content)
if str(BASE) not in sys.path: sys.path.insert(0, str(BASE))
from preflight import host_check
host_check(BASE, before_install=True)
print('硬件前置检查通过。未加载任何模型。')


## 2 · 安装固定环境与官方源码
首次安装需要网络并下载较大的 PyTorch wheel。源码固定在官方 TI2V 发布时期的提交，避免后续动画/语音任务引入额外依赖。
FlashAttention 使用 Dao-AILab 官方发布的 Python 3.11 / PyTorch 2.6 / CUDA 12 / ABI=false 预编译 wheel，**不在 Colab 编译 CUDA 扩展**。
如下载失败或小算子失败，先修复环境；不要跳过 FlashAttention。此版本 Wan 模型会直接调用它。
顶层依赖固定，完整实际版本会保存为 `environment.freeze.txt`；不声称所有间接依赖都完全锁定。


In [ ]:
WAN_COMMIT = 'e5058c08fada388b04bb3d1a4734bddbd2fdeac5'
FLASH_WHEEL = 'https://github.com/Dao-AILab/flash-attention/releases/download/v2.7.4.post1/flash_attn-2.7.4.post1%2Bcu12torch2.6cxx11abiFALSE-cp311-cp311-linux_x86_64.whl'
PY = BASE / 'venv/bin/python'
REPO = BASE / 'Wan2.2'
# 仅在 Colab 主环境安装引导器；模型依赖全部放入独立环境。
run_logged([sys.executable, '-m', 'pip', 'install', '--disable-pip-version-check',
            'uv==0.6.17'], 'bootstrap')
if not PY.exists():
    run_logged([sys.executable, '-m', 'uv', 'venv', '--python', '3.11.11',
                '--seed', BASE / 'venv'], 'venv')
run_logged([PY, '-m', 'pip', 'install', 'torch==2.6.0', 'torchvision==0.21.0',
            '--index-url', 'https://download.pytorch.org/whl/cu124'], 'install_torch')
run_logged([PY, '-m', 'pip', 'install', '-r', BASE / 'requirements.txt'], 'install_deps')
run_logged([PY, '-m', 'pip', 'install', '--no-deps', FLASH_WHEEL], 'install_flash')
run_logged([PY, '-m', 'pip', 'check'], 'pip_check')
if not shutil.which('ffprobe'):
    run_logged(['apt-get', 'update', '-qq'], 'apt_update')
    run_logged(['apt-get', 'install', '-y', '-qq', 'ffmpeg'], 'apt_ffmpeg')
if not REPO.exists():
    run_logged(['git', 'clone', '--filter=blob:none', '--no-checkout',
                'https://github.com/Wan-Video/Wan2.2.git', REPO], 'clone_wan')
if subprocess.check_output(['git', '-C', str(REPO), 'remote', 'get-url', 'origin'], text=True).strip() != 'https://github.com/Wan-Video/Wan2.2.git':
    raise RuntimeError('源码目录的来源不匹配，请换干净运行时')
run_logged(['git', '-C', REPO, 'checkout', '--detach', WAN_COMMIT], 'checkout_wan')
assert subprocess.check_output(['git','-C',str(REPO),'rev-parse','HEAD'],text=True).strip() == WAN_COMMIT
if subprocess.check_output(['git','-C',str(REPO),'status','--porcelain'],text=True).strip():
    raise RuntimeError('官方源码存在本地修改，请使用干净目录')
(BASE / 'environment.freeze.txt').write_text(subprocess.check_output([str(PY), '-m', 'pip', 'freeze'], text=True))
cfg = dict(wan_commit=WAN_COMMIT, verified_on='2026-11-26',
           model_specs={'wan': {'repo': 'Wan-AI/Wan2.2-TI2V-5B', 'revision': '921dbaf3f1674a56f47e83fb80a34bac8a8f203e'}, 'text': {'repo': 'Qwen/Qwen2.5-1.5B-Instruct', 'revision': '989aa7980e4cf806f80c7fef2b1adb7bc71aa306'}, 'image': {'repo': 'stabilityai/sd-turbo', 'revision': 'b261bac6fd2cf515557d5d0707481eafa0485ec2'}}, models={})
# 重新安装环境后，重新运行下载单元格以核对并恢复模型目录。
(BASE / 'config.json').write_text(json.dumps(cfg, ensure_ascii=False, indent=2))
print('安装完成。无需重启 Colab 主内核。')


## 3 · 导入、CLI、CUDA 与 FlashAttention 自检
这一步不下载模型。失败时请停下，查看日志；不要带着不完整依赖继续。


In [ ]:
env = dict(os.environ, PYTHONPATH=str(REPO)+os.pathsep+str(BASE))
run_logged([PY, BASE / 'preflight.py'], 'gpu_selfcheck', env=env, cwd=REPO)
help_text = subprocess.check_output([str(PY), str(REPO/'generate.py'), '--help'],
                                   cwd=REPO, env=env, text=True)
for flag in ['--task', '--image', '--frame_num', '--save_file', '--offload_model',
             '--convert_model_dtype', '--t5_cpu', '--sample_steps', '--sample_solver']:
    assert flag in help_text, '官方 CLI 缺少参数 '+flag
(BASE / 'generate-help.txt').write_text(help_text)
print('官方 CLI 参数检查通过。')


## 4 · 下载并核验固定版本权重
三个模型均来自各自官方组织，按提交 SHA 下载。下载使用缓存，可重试；逐文件检查长度，记录清单。
全量 Wan 权重含大型 CPU 文本编码器，下载量是数十 GB，并非只有 5B 视频模型本体。
本节会确保独立环境已安装 `hf-xet==1.8.0`，启用 Xet 下载，避免退回普通 HTTP。
SD-Turbo 只下载此工作流需要的 FP16 权重，跳过重复 FP32 与单文件版本。使用 4 个文件下载线程，不默认启用高内存的高性能模式。
重新运行会复用已完成缓存；HTTP/Xet 切换后，未完成的大文件可能重新下载，不保证跨协议断点续传。
默认无需 Hugging Face Token。如遇速率限制，可在 Colab「密钥」里添加 `HF_TOKEN` 并授权此笔记本；不会保存到配置文件。
模型按各自模型卡许可使用，尤其 SD-Turbo 商业用途请检查当前 Stability AI 许可。


In [ ]:
run_logged([PY, '-m', 'pip', 'install', 'hf-xet==1.8.0'], 'install_xet')
download_env = dict(os.environ)
download_env.pop('HF_HUB_DISABLE_XET', None)
download_env['HF_XET_CACHE'] = str(BASE / 'hf_cache' / 'xet')
try:
    from google.colab import userdata
    token = userdata.get('HF_TOKEN')
    if token: download_env['HF_TOKEN'] = token
except Exception:
    pass
try:
    run_logged([PY, BASE / 'download_models.py'], 'download_models', env=download_env)
finally:
    download_env.pop('HF_TOKEN', None)
    token = None
print('磁盘剩余 GiB:', round(shutil.disk_usage(BASE).free/2**30, 1))


## 5 · 想法 → 文案与提示词
输出中文文案、英文首帧提示词、英文运动提示词。原始模型文字也会保存。
若模型未返回有效 JSON，会明确失败，不会偷偷换成预设提示词。可参考下方修正单元格手工修复。
每个阶段最多运行两小时；超时或中断会终止对应进程组，保留日志。


In [ ]:
from core import new_run, run_stage, read_plan
RUN = new_run(IDEA, SEED, FRAMES, STEPS)
print('本次运行目录:', RUN)
print('开始生成文案；此单元格等待时，可在文件面板查看 text.log。')
run_stage('text', RUN)
PLAN = read_plan(RUN)
print(PLAN['copy_zh'])
display(JSON(PLAN))
display(JSON(json.loads((RUN / 'text_metrics.json').read_text())))


### 可选 · 审阅/修改提示词
只有将 `EDIT_PLAN=True` 才会写回。失败时可先查看 `(RUN/'text_raw.txt').read_text()`，用实际想法修正三项内容。
图片与运动提示词请保持同一主体/环境；尽量使用单镜头、缓慢动作。此文案不会自动变成配音或视频字幕。


In [ ]:
EDIT_PLAN = False
if EDIT_PLAN:
    PLAN = {
        'copy_zh': '请填写中文文案',
        'image_prompt': 'Write the English first-frame description here.',
        'video_prompt': 'Describe the same scene with gentle movement in English.'
    }
    (RUN / 'plan.json').write_text(json.dumps(PLAN, ensure_ascii=False, indent=2))
    display(JSON(read_plan(RUN)))


## 6 · 文生图与实际视频首帧展示
SD-Turbo 官方建议的基础分辨率为 512×512。这里先生成原图，再居中裁切为横向构图并放大为 1280×704；放大不增加真实细节。
两张图均保留。审阅实际首帧后，再运行下一步；若裁切不理想，可修改提示词并重跑此阶段。


In [ ]:
run_stage('image', RUN)
print('原始生成图：')
display(Image(filename=str(RUN / 'image_original.png'), width=512))
print('送入 Wan 的实际首帧（裁切并放大）：')
display(Image(filename=str(RUN / 'image.png'), width=768))
display(JSON(json.loads((RUN / 'image_metrics.json').read_text())))


## 7 · 官方 Wan 图生视频
使用 `ti2v-5B`，不是 A14B。保留官方显存卸载组合：`offload_model=True`、`convert_model_dtype`、`t5_cpu`。
使用固定输出路径，无文件轮询猜测、无 glob 列表合并错误、无 ComfyUI 节点依赖。
默认 33 帧 / 20 步是试跑参数，先不要提高。运行中在文件面板查看 `video.log`；有进度条但不保证实时百分比。


In [ ]:
from core import video_command
cmd = video_command(RUN)
print('官方任务:', cmd[cmd.index('--task')+1], '| 帧数:', FRAMES, '| 步数:', STEPS)
print('请等待；本次日志:', RUN / 'video.log')
run_stage('video', RUN)
display(Video(filename=str(RUN / 'video.mp4'), embed=True, width=768))
display(JSON(json.loads((RUN / 'video_info.json').read_text())))
display(JSON(json.loads((RUN / 'video_metrics.json').read_text())))


## 8 · 下载本次结果
打包文案、首帧、视频、日志和实际环境记录，不打包权重、密码或 Token。
保存结果后再断开运行时。模型采样随机性、驱动与硬件差异可能使同 seed 输出不同。


In [ ]:
import zipfile
archive = BASE / ('results_' + RUN.name + '.zip')
with zipfile.ZipFile(archive, 'w', zipfile.ZIP_DEFLATED) as z:
    for p in RUN.iterdir():
        if p.is_file(): z.write(p, 'run/' + p.name)
    for name in ['config.json','environment.freeze.txt','model_manifest.json','gpu_selfcheck.log']:
        p = BASE / name
        if p.exists(): z.write(p, 'environment/' + name)
from google.colab import files
files.download(str(archive))


## 9 · 可选：启动带鉴权的网页和异步 API（默认关闭）
将第一节的 `ENABLE_WEB` 改为 `True`，或在此单元格前赋值，然后运行本节。先完成第 1–4 节。
服务只监听 `127.0.0.1:8000`，不开启 Gradio share，不自动创建公网链接。
用户名为 `owner`；手工输入至少 16 字符密码（隐藏输入，不写入文件）。所有路由和媒体文件均受鉴权保护。
一个任务最多两小时/阶段；同一时刻只接受一个任务，额外请求返回 409，不排无限队列。

API：`POST /jobs` 返回 202 和任务 ID；`GET /jobs/{id}` 返回阶段/文案/就绪状态；
`GET /files/{id}/image.png` 和 `video.mp4` 获取结果；`GET /health` 检查服务。
全部请求使用 HTTP Basic；公网只使用 Cloudflare 生成的 HTTPS 链接。后台任务避免长请求超时，轮询避免 SSE 限制。
同一账号可读取该服务会话中的所有产物，仅用于个人临时演示，不提供多租户隔离或抗暴力破解保证。


In [ ]:
SERVER = globals().get('SERVER', None)
WEB_AUTH = globals().get('WEB_AUTH', None)
def http_get(url, auth=None):
    request = urllib.request.Request(url)
    if auth:
        value = base64.b64encode((auth[0]+':'+auth[1]).encode()).decode()
        request.add_header('Authorization', 'Basic '+value)
    return urllib.request.urlopen(request, timeout=15)
if ENABLE_WEB:
    if SERVER is not None and SERVER.poll() is None:
        print('服务已在运行。修改密码前请先运行停止单元格。')
    else:
        password = getpass.getpass('设置本次网页密码（至少 16 字符，不会回显）：')
        if len(password) < 16: raise ValueError('密码至少 16 字符')
        WEB_AUTH = ('owner', password)
        server_env = dict(os.environ, LAB_USER='owner', LAB_PASSWORD=password)
        password = None
        server_log = (BASE / 'server.log').open('w')
        try:
            SERVER = subprocess.Popen([str(PY), str(BASE/'server.py')], env=server_env,
                cwd=BASE, stdout=server_log, stderr=subprocess.STDOUT, start_new_session=True)
        finally:
            server_log.close(); server_env.pop('LAB_PASSWORD', None)
        try:
            for _ in range(60):
                if SERVER.poll() is not None: raise RuntimeError('服务退出，查看 server.log')
                try:
                    with http_get('http://127.0.0.1:8000/health', WEB_AUTH) as r:
                        assert r.status == 200
                    break
                except (OSError, urllib.error.URLError): time.sleep(0.5)
            else: raise RuntimeError('服务启动超时')
            try:
                http_get('http://127.0.0.1:8000/health')
                raise RuntimeError('鉴权未生效，拒绝继续')
            except urllib.error.HTTPError as e:
                assert e.code == 401
            print('网页/API已在本地启动；匿名访问 401，自带密码访问 200。')
            print('浏览网页可继续运行下一节 Cloudflare；未开启任何公网隧道。')
        except BaseException:
            os.killpg(SERVER.pid, signal.SIGTERM); SERVER.wait(); WEB_AUTH = None
            raise
else:
    print('网页服务未启用。')


## 10 · 可选：Cloudflare 临时公网入口（默认关闭）
设置 `ENABLE_PUBLIC=True` 后运行。仅当本地鉴权自检通过时才启动；不需要 Cloudflare 账号/域名。
固定官方 cloudflared 发布版本，并校验发布 SHA256。隧道不是鉴权，密码由 FastAPI 校验。
Quick Tunnel 是临时测试工具，无可用性保证；链接随进程停止失效。这里只用普通 HTTP 轮询，不使用 SSE。
先验证本地服务，再依次尝试 auto 和 HTTP/2，每种协议最多等待 120 秒；输出具体 HTTP 状态或连接错误。
只有收到本应用的匿名 401 与带密码的健康 JSON 200 才视为成功；不会跳过鉴权检查。
若组织禁用隧道或 Cloudflare 不可达，请继续使用笔记本；不要为绕过限制取消鉴权。


In [ ]:
# Cloudflare recovery cell: only restarts this notebook's tunnel, never the model server.
import base64
import hashlib
import json
import os
from pathlib import Path
import re
import signal
import subprocess
import time
import urllib.request
import urllib.error

CF_SHA256 = '56f13335143ec0e22772963290683a84a04114a35ec64cbc794a4d39ff9bb8cf'
CF_BINARY_URL = 'https://github.com/cloudflare/cloudflared/releases/download/2026.11.2/cloudflared-linux-amd64'

class NoRedirect(urllib.request.HTTPRedirectHandler):
    def redirect_request(self, req, fp, code, msg, headers, newurl):
        return None  # Never forward credentials to a redirected host.

def probe_health(url, auth=None, timeout=5):
    headers = {'Accept': 'application/json', 'User-Agent': 'Wan-L4-healthcheck/2'}
    if auth:
        headers['Authorization'] = 'Basic ' + base64.b64encode(
            (auth[0] + ':' + auth[1]).encode()).decode()
    request = urllib.request.Request(url.rstrip('/') + '/health', headers=headers)
    try:
        with urllib.request.build_opener(NoRedirect()).open(request, timeout=timeout) as response:
            code, body = response.status, response.read(4096)
            challenge = response.headers.get('WWW-Authenticate', '')
    except urllib.error.HTTPError as error:
        code, body = error.code, error.read(4096)
        challenge = error.headers.get('WWW-Authenticate', '')
    except (OSError, urllib.error.URLError) as error:
        return {'code': None, 'ok': False, 'app401': False,
                'detail': type(error).__name__ + ': ' + str(error)}
    try:
        data = json.loads(body)
    except (ValueError, UnicodeError):
        data = {}
    return {'code': code, 'ok': code == 200 and isinstance(data, dict) and data.get('ok') is True,
        'app401': code == 401 and 'Basic' in challenge and isinstance(data, dict)
                  and data.get('detail') == 'Authentication required',
        'detail': 'HTTP ' + str(code)}

def classify_probes(anonymous, authenticated):
    if anonymous['code'] is not None and 200 <= anonymous['code'] < 300:
        return 'unsafe', '匿名请求意外成功，不能公开该服务'
    if anonymous['app401'] and authenticated['ok']:
        return 'ready', '匿名 401 / 带密码 200，且健康响应内容正确'
    if anonymous['app401'] and authenticated['code'] == 401:
        return 'auth', '已到达应用，但带密码请求仍为 401：重跑第 9 节以同步服务密码与 WEB_AUTH'
    return 'retry', '匿名=' + anonymous['detail'] + '；带密码=' + authenticated['detail']

def stop_tunnel(proc):
    if proc is not None and proc.poll() is None:
        os.killpg(proc.pid, signal.SIGTERM)
        try:
            proc.wait(timeout=10)
        except subprocess.TimeoutExpired:
            os.killpg(proc.pid, signal.SIGKILL)
            proc.wait()

def recover_tunnel(base, auth, old_tunnel=None, seconds_per_protocol=120):
    base = Path(base)
    if not auth or len(auth) != 2:
        raise RuntimeError('找不到 WEB_AUTH，请先运行第 9 节启动带密码服务')
    # Validate the origin before creating any public exposure.
    local_anonymous = probe_health('http://127.0.0.1:8000')
    local_authenticated = probe_health('http://127.0.0.1:8000', auth)
    status, detail = classify_probes(local_anonymous, local_authenticated)
    print('本地自检：', detail, flush=True)
    if status != 'ready':
        raise RuntimeError('本地服务未通过检查；先处理第 9 节。' + detail)
    binary = base / 'cloudflared'
    if not binary.exists() or hashlib.sha256(binary.read_bytes()).hexdigest() != CF_SHA256:
        temp = binary.with_suffix('.download')
        urllib.request.urlretrieve(CF_BINARY_URL, temp)
        if hashlib.sha256(temp.read_bytes()).hexdigest() != CF_SHA256:
            temp.unlink()
            raise RuntimeError('cloudflared SHA256 不匹配')
        temp.replace(binary)
    binary.chmod(0o700)
    if any((Path.home()/'.cloudflared'/name).exists() for name in ('config.yml', 'config.yaml')):
        raise RuntimeError('存在 cloudflared 配置，请在干净运行时处理，避免使用已有隧道配置')
    stop_tunnel(old_tunnel)
    last_detail = '没有获取到临时地址'
    # Preserve auto (QUIC with built-in fallback); HTTP/2 is a second diagnostic attempt.
    for protocol in ('auto', 'http2'):
        log_path = base / ('tunnel_' + protocol + '.log')
        print('启动 Cloudflare，协议=' + protocol + '；日志=' + str(log_path), flush=True)
        command = [str(binary), 'tunnel', '--no-autoupdate', '--protocol', protocol,
            '--url', 'http://127.0.0.1:8000']
        with log_path.open('w') as log:
            proc = subprocess.Popen(command, stdout=log, stderr=subprocess.STDOUT,
                                    start_new_session=True)
        success = False
        url, previous_detail = None, None
        deadline = time.monotonic() + seconds_per_protocol
        try:
            while time.monotonic() < deadline:
                if proc.poll() is not None:
                    last_detail = 'cloudflared 已退出，exit=' + str(proc.returncode)
                    break
                log_text = log_path.read_text(errors='replace')
                match = re.search(r'https://[a-z0-9-]+\.trycloudflare\.com', log_text)
                if match:
                    url = match.group(0)
                    # A printed URL is not proof that the tunnel is connected.
                    anonymous = probe_health(url)
                    authenticated = probe_health(url, auth)
                    status, last_detail = classify_probes(anonymous, authenticated)
                    if status == 'ready':
                        success = True
                        print('公网自检通过：' + last_detail, flush=True)
                        return proc, url
                    if status in ('unsafe', 'auth'):
                        raise RuntimeError(last_detail)
                    if last_detail != previous_detail:
                        print('等待隧道/DNS就绪：' + last_detail, flush=True)
                        previous_detail = last_detail
                time.sleep(2)
            print('本次未就绪：' + last_detail, flush=True)
            # Keep full logs local; users can share the relevant error lines without credentials.
            print('请查看日志最后约 30 行：' + str(log_path), flush=True)
        finally:
            if not success:
                stop_tunnel(proc)
    raise RuntimeError('公网未就绪：' + last_detail +
        '。已停止失败隧道，本地服务和模型结果仍保留；查看 tunnel_http2.log / tunnel_auto.log。')

if globals().get('ENABLE_PUBLIC', False):
    # Clear stale URL even when recovery raises. Do not print WEB_AUTH.
    PUBLIC_URL = None
    _old_tunnel = globals().get('TUNNEL')
    TUNNEL = None
    TUNNEL, PUBLIC_URL = recover_tunnel(BASE, globals().get('WEB_AUTH'), _old_tunnel)
    print('页面地址：', PUBLIC_URL)
    print('用户名 owner；使用第 9 节设置的密码。')
else:
    print('公网隧道未启用。需要使用时设置 ENABLE_PUBLIC=True。')


### API 调用示例（默认不提交）
先启动服务。示例从内存取密码，不会把密码或带密码 URL 打印到输出。
外部客户端使用自己的密码管理器/交互输入传入 Basic Auth；不要把密码写进 URL、公开代码或笔记本。


In [ ]:
SUBMIT_API_EXAMPLE = False
if SUBMIT_API_EXAMPLE:
    if not WEB_AUTH: raise RuntimeError('先启动网页服务')
    url = PUBLIC_URL or 'http://127.0.0.1:8000'
    request = urllib.request.Request(url+'/jobs', method='POST',
        data=json.dumps(dict(idea=IDEA, seed=SEED, frames=FRAMES, steps=STEPS)).encode(),
        headers={'Content-Type': 'application/json',
                 'Authorization': 'Basic '+base64.b64encode((':'.join(WEB_AUTH)).encode()).decode()})
    with urllib.request.urlopen(request, timeout=20) as r:
        job = json.load(r)
    print(job)  # 202：随后 GET /jobs/{id} 轮询，每 4 秒一次即可。


## 11 · 停止网页、隧道及其模型子进程
需要停止时才运行此节。网页任务的模型子进程使用独立进程组，按运行目录精确匹配后终止；不会结束其他用户进程。
不自动删除权重和结果。若笔记本阶段因中断未正常结束，可断开并删除 Colab 运行时彻底释放资源。


In [ ]:
STOP_SERVICES = False  # 需要停止时改为 True
if STOP_SERVICES:
    # Stop accepting new work before terminating active stage workers.
    for name in ('TUNNEL','SERVER'):
        p = globals().get(name)
        if p is not None and p.poll() is None:
            os.killpg(p.pid, signal.SIGTERM)
            try: p.wait(timeout=10)
            except subprocess.TimeoutExpired:
                os.killpg(p.pid, signal.SIGKILL); p.wait()
    for proc in Path('/proc').iterdir():
        if not proc.name.isdigit(): continue
        try:
            args = (proc/'cmdline').read_bytes().split(b'\0')
            text = [a.decode(errors='replace') for a in args if a]
            is_worker = str(BASE/'worker.py') in text or str(REPO/'generate.py') in text
            is_our_run = any(a.startswith(str(BASE/'runs')+'/') for a in text)
            pid = int(proc.name)
            if is_worker and is_our_run and os.getpgid(pid) == pid:
                os.killpg(pid, signal.SIGTERM)
                time.sleep(0.2)
                try: os.killpg(pid, signal.SIGKILL)
                except ProcessLookupError: pass
        except (FileNotFoundError, ProcessLookupError, PermissionError): pass
    WEB_AUTH = None
    SERVER = TUNNEL = PUBLIC_URL = None
    print('网页、隧道及匹配的模型进程已停止；结果仍保留在 /content。')
else:
    print('未请求停止服务。')


## 故障处理与验证边界
- **CUDA OOM**：确认 L4 空闲；不要同时运行其他 GPU 单元格；保留三项官方卸载参数；先用 33 帧。
  若短片仍失败，查看日志中的具体分配位置，重启干净运行时重试；不要宣称 24GB 标签保证所有参数能运行。
- **进程 exit=-9 / RAM 不足**：CPU T5 和初始模型加载需要大量 RAM，确认高内存 64GB，至少 40 GiB 可用。
- **FlashAttention 导入/算子失败**：核对 Python 3.11、torch 2.6.0+cu124、ABI=false、驱动；保持干净独立环境。
  本笔记本不会擅自把官方注意力函数替换为忽略 mask 的实现。
- **下载中断/401/403**：检查 Hugging Face 可达性和模型访问权限；可重跑第 4 节使用缓存。
- **文案 JSON 失败**：查看 text_raw.txt，修正 plan.json；再次运行 read_plan 后继续图片阶段。
- **视频失败**：首帧和文案仍保留，可单独重跑第 7 节。每次调用覆盖该阶段日志，重试前可先备份。
- **网页 409**：已有任务或笔记本正在使用 GPU，等待完成；网页失败阶段可在本地日志中排查。
- **Tunnel 502/无法打开**：确认服务尚在运行，检查 server.log / tunnel.log；可关闭公网继续在笔记本展示。
- **帧/时长**：33/65/121 均为 4n+1；按官方 24 FPS 输出。ffprobe 核对尺寸、帧率和帧数。
  imageio ffmpeg 编码可能将少量 RGB 值转换为视频色彩空间，不保证第一帧逐像素相同。

**未实际验证：L4 端到端生成、峰值显存、速度、Colab 当日镜像/网络、真实公网隧道连接、最终画质。**
固定版本减少上游变更带来的不确定性，但未来 Colab 驱动/网络变化仍可能需要维护。
这不是生产部署，不含语音、字幕、多镜头剪辑、GPU 抢占或保活逻辑。

## 官方资料（核对日期：2026-11-26）
1. [Wan 官方仓库与 24GB 卸载命令](https://github.com/Wan-Video/Wan2.2)
2. [本笔记本固定官方源码提交](https://github.com/Wan-Video/Wan2.2/tree/e5058c08fada388b04bb3d1a4734bddbd2fdeac5)
3. [Wan 官方 TI2V-5B 模型卡](https://huggingface.co/Wan-AI/Wan2.2-TI2V-5B)
4. [Qwen 1.5B 官方模型卡与 chat template](https://huggingface.co/Qwen/Qwen2.5-1.5B-Instruct)
5. [SD-Turbo 官方模型卡与 512/CFG=0 用法](https://huggingface.co/stabilityai/sd-turbo)
6. [PyTorch 官方历史安装版本](https://pytorch.org/get-started/previous-versions/)
7. [FlashAttention 官方 wheel 发布](https://github.com/Dao-AILab/flash-attention/releases/tag/v2.7.4.post1)
8. [FastAPI 官方 HTTP Basic Auth](https://fastapi.tiangolo.com/advanced/security/http-basic-auth/)
9. [Cloudflare Quick Tunnel 与 SSE 限制](https://developers.cloudflare.com/tunnel/get-started/quick-tunnels/)
10. [固定 cloudflared 官方发布](https://github.com/cloudflare/cloudflared/releases/tag/2026.11.2)
